# LAD XML 导入测试

本 Notebook 对各类 LAD XML 做端到端测试：  
1. 启动 TIA Portal
2. 新建项目，添加 S7-1200 CPU 1214C
3. 创建测试所需的全局 DB 和变量表
4. 逐条导入测试 XML，记录 通过 / 报错
5. 打印汇总结果

> **运行前提**：以 **管理员权限** 启动 VS Code/Jupyter Kernel（TIA Openness 要求），  
> 且 TIA Portal V17 已安装于默认路径。

## 0. 路径初始化

In [1]:
import sys, os

# 将工作区根目录加入路径
ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__') if '__file__' in dir() else '.', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('工作区根目录:', ROOT)

工作区根目录: e:\PlcProject\Code\PLC\SCDW


## 1. 启动 TIA Portal

In [2]:
from openness.tia_core import start_tia_portal

# with_ui=True：有界面模式，方便观察导入结果；False：无界面更快
tia = start_tia_portal(with_ui=True)
print('TIA Portal 已启动:', tia)

InternalPythonnetException: Failed to create Python type for Siemens.Engineering.TiaPortalProcess ---> System.IO.FileNotFoundException: 未能加载文件或程序集“Siemens.Engineering.Contract, Version=1700.4.401.1, Culture=neutral, PublicKeyToken=37a18b206f7724a6”或它的某一个依赖项。系统找不到指定的文件。
   在 System.Signature.GetSignature(Void* pCorSig, Int32 cCorSig, RuntimeFieldHandleInternal fieldHandle, IRuntimeMethodInfo methodHandle, RuntimeType declaringType)
   在 System.Reflection.RuntimeConstructorInfo.GetParametersNoCopy()
   在 System.Reflection.MethodBase.GetParameterTypes()
   在 System.Reflection.MethodBase.FormatNameAndSig(Boolean serialization)
   在 System.Reflection.RuntimeConstructorInfo.ToString()
   在 Python.Runtime.MethodObject..ctor(MaybeType type, String name, MethodBase[] info, Boolean allow_threads, Boolean argsReversed)
   在 Python.Runtime.ClassManager.GetClassInfo(Type type, ClassBase impl)
   在 Python.Runtime.ClassManager.InitClassBase(Type type, ClassBase impl, ReflectedClrType pyType)

## 2. 新建项目并添加 S7-1200 CPU 1214C

In [ ]:
from openness.tia_core import create_project, save_project
from openness.tia_hardware import add_plc_device

PROJECT_ROOT = r'E:\PlcProject\TestProjects'
PROJECT_NAME = 'LAD_XML_Test'

# overwrite=True：每次运行都重建，保持干净测试环境
project = create_project(tia, PROJECT_ROOT, PROJECT_NAME, overwrite=True)
print('项目已创建:', project.Name)

# S7-1214C AC/DC/Rly V4.4（根据本地 TIA 版本调整 /V 号）
CPU_ORDER = 'OrderNumber:6ES7 214-1BG40-0XB0/V4.4'
device, plc_sw = add_plc_device(project, CPU_ORDER, 'PLC_1', 'PLC_1')
print('PLC 设备已添加:', device.Name, '| PLC软件对象:', plc_sw)

## 3. 准备公共测试数据（全局 DB + 变量表）

In [ ]:
from openness.tia_blocks import create_global_db
from openness.tia_blocks import DBVariable
from openness.tia_tags import create_tag_table, add_tag

# 全局 DB：DB1，包含常用测试变量
db_vars = [
    DBVariable('MotorRun',    'Bool',  'false', '电机运行状态'),
    DBVariable('MotorFault',  'Bool',  'false', '电机故障'),
    DBVariable('StartBtn',    'Bool',  'false', '启动按钮'),
    DBVariable('StopBtn',     'Bool',  'false', '停止按钮'),
    DBVariable('Counter',     'Int',   '0',     '计数器'),
    DBVariable('SetPoint',    'Real',  '0.0',   '设定值'),
    DBVariable('PV',          'Real',  '0.0',   '过程值'),
    DBVariable('Timer1',      'IEC_TIMER', '', '定时器实例'),
]
create_global_db(plc_sw, 'TestDB', db_vars, db_number=1)
print('DB1 TestDB 已创建')

# 变量表：直接寻址测试
tt = create_tag_table(plc_sw, 'TestTags')
add_tag(tt, 'Sensor1', 'Bool', '%I0.0', '传感器1')
add_tag(tt, 'Sensor2', 'Bool', '%I0.1', '传感器2')
add_tag(tt, 'Output1', 'Bool', '%Q0.0', '输出1')
print('变量表 TestTags 已创建')

save_project(project)
print('项目已保存')

## 4. 测试用例定义

每个 `(名称, XML字符串)` 对应一条测试。  
**预期通过**的测试名以 `PASS_` 开头，**预期报错**的以 `FAIL_` 开头（用于验证预处理检测器是否正常工作）。

In [ ]:
# ────────────────────────────────────────────────────────────
# 测试用 XML 公共头/尾
# ────────────────────────────────────────────────────────────
def _wrap_fc(name: str, number: int, networks_xml: str) -> str:
    """将 FlgNet 片段包装成完整的 FC SimaticML XML。"""
    return f"""<?xml version="1.0" encoding="utf-8"?>
<Document>
  <Engineering version="V17" />
  <SW.Blocks.FC ID="0">
    <AttributeList>
      <AutoNumber>false</AutoNumber>
      <HeaderAuthor />
      <HeaderFamily />
      <HeaderName />
      <HeaderVersion>1.0</HeaderVersion>
      <Interface><Sections xmlns="http://www.siemens.com/automation/Openness/SW/Interface/v5"><Section Name="Input" /><Section Name="Output" /><Section Name="InOut" /><Section Name="Temp" /><Section Name="Constant" /></Sections></Interface>
      <Name>{name}</Name>
      <Number>{number}</Number>
      <ProgrammingLanguage>LAD</ProgrammingLanguage>
    </AttributeList>
    <ObjectList>
{networks_xml}
    </ObjectList>
  </SW.Blocks.FC>
</Document>"""


def _net(uid_base: int, title: str, flgnet_xml: str) -> str:
    """包装单个 CompileUnit。"""
    return f"""      <SW.Blocks.CompileUnit ID="{uid_base}" CompositionName="Networks">
        <AttributeList>
          <NetworkSource>
            <FlgNet xmlns="http://www.siemens.com/automation/Openness/SW/NetworkSource/FlgNet/v4">
{flgnet_xml}
            </FlgNet>
          </NetworkSource>
          <ProgrammingLanguage>LAD</ProgrammingLanguage>
        </AttributeList>
        <ObjectList>
          <MultilingualText ID="{uid_base+1}" CompositionName="Comment">
            <ObjectList>
              <MultilingualTextItem ID="{uid_base+2}" CompositionName="Items">
                <AttributeList><Culture>zh-CN</Culture><Text>{title}</Text></AttributeList>
              </MultilingualTextItem>
            </ObjectList>
          </MultilingualText>
        </ObjectList>
      </SW.Blocks.CompileUnit>"""


print('XML 构造辅助函数已定义')

In [ ]:
# ────────────────────────────────────────────────────────────
# 测试用例列表
# ────────────────────────────────────────────────────────────
NS = 'xmlns="http://www.siemens.com/automation/Openness/SW/NetworkSource/FlgNet/v4"'

TEST_CASES = []

# ── PASS 1：简单串联 Contact → Coil ──────────────────────────
TEST_CASES.append(('PASS_串联Contact_Coil', _wrap_fc('TC_Serial', 1, _net(10, '串联触点线圈', """
              <Parts>
                <Contact UId="21">
                  <Access UId="22" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StartBtn" /></Symbol>
                  </Access>
                </Contact>
                <Coil UId="23">
                  <Access UId="24" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorRun" /></Symbol>
                  </Access>
                </Coil>
              </Parts>
              <Wires>
                <Wire UId="25"><Powerrail /><NameCon UId="21" Name="in" /></Wire>
                <Wire UId="26"><NameCon UId="21" Name="out" /><NameCon UId="23" Name="in" /></Wire>
                <Wire UId="27"><NameCon UId="23" Name="out" /></Wire>
              </Wires>
"""))))

# ── PASS 2：并联支路（正确写法：同一 Powerrail Wire 多 NameCon）──
TEST_CASES.append(('PASS_并联支路正确写法', _wrap_fc('TC_Parallel', 2, _net(30, '并联正确写法', """
              <Parts>
                <Contact UId="101">
                  <Access UId="102" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StartBtn" /></Symbol>
                  </Access>
                </Contact>
                <SCoil UId="103">
                  <Access UId="104" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorRun" /></Symbol>
                  </Access>
                </SCoil>
                <Contact UId="201">
                  <Access UId="202" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StopBtn" /></Symbol>
                  </Access>
                </Contact>
                <RCoil UId="203">
                  <Access UId="204" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorRun" /></Symbol>
                  </Access>
                </RCoil>
              </Parts>
              <Wires>
                <Wire UId="401">
                  <Powerrail />
                  <NameCon UId="101" Name="in" />
                  <NameCon UId="201" Name="in" />
                </Wire>
                <Wire UId="402"><NameCon UId="101" Name="out" /><NameCon UId="103" Name="in" /></Wire>
                <Wire UId="403"><NameCon UId="103" Name="out" /></Wire>
                <Wire UId="404"><NameCon UId="201" Name="out" /><NameCon UId="203" Name="in" /></Wire>
                <Wire UId="405"><NameCon UId="203" Name="out" /></Wire>
              </Wires>
"""))))

# ── PASS 3：同一变量正确多处引用（各自用不同 Access UId）──────
TEST_CASES.append(('PASS_IdentCon多处引用正确写法', _wrap_fc('TC_MultiRef', 3, _net(50, 'IdentCon多处正确引用', """
              <Parts>
                <Contact UId="301">
                  <Access UId="302" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StartBtn" /></Symbol>
                  </Access>
                </Contact>
                <Contact UId="303">
                  <Access UId="304" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StartBtn" /></Symbol>
                  </Access>
                </Contact>
                <Coil UId="305">
                  <Access UId="306" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorRun" /></Symbol>
                  </Access>
                </Coil>
                <Coil UId="307">
                  <Access UId="308" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorFault" /></Symbol>
                  </Access>
                </Coil>
              </Parts>
              <Wires>
                <Wire UId="401">
                  <Powerrail />
                  <NameCon UId="301" Name="in" />
                  <NameCon UId="303" Name="in" />
                </Wire>
                <Wire UId="402"><NameCon UId="301" Name="out" /><NameCon UId="305" Name="in" /></Wire>
                <Wire UId="403"><NameCon UId="305" Name="out" /></Wire>
                <Wire UId="404"><NameCon UId="303" Name="out" /><NameCon UId="307" Name="in" /></Wire>
                <Wire UId="405"><NameCon UId="307" Name="out" /></Wire>
              </Wires>
"""))))

# ── PASS 4：DisabledENO 自动剥除（加了属性但系统会剥除）────────
TEST_CASES.append(('PASS_DisabledENO自动剥除', _wrap_fc('TC_DisabledENO', 4, _net(70, 'DisabledENO自动剥除', """
              <Parts>
                <Contact UId="21" DisabledENO="true">
                  <Access UId="22" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StartBtn" /></Symbol>
                  </Access>
                </Contact>
                <Coil UId="23" DisabledENO="true">
                  <Access UId="24" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorRun" /></Symbol>
                  </Access>
                </Coil>
              </Parts>
              <Wires>
                <Wire UId="25"><Powerrail /><NameCon UId="21" Name="in" /></Wire>
                <Wire UId="26"><NameCon UId="21" Name="out" /><NameCon UId="23" Name="in" /></Wire>
                <Wire UId="27"><NameCon UId="23" Name="out" /></Wire>
              </Wires>
"""))))

# ── PASS 5：Part 排序错误自动修复（Contact堆前/Coil堆后）──────
# 系统会自动拓扑排序修复，应该能成功导入
TEST_CASES.append(('PASS_Part排序错误自动修复', _wrap_fc('TC_PartOrder', 5, _net(90, 'Part排序错误自动修复', """
              <Parts>
                <Contact UId="101">
                  <Access UId="102" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StartBtn" /></Symbol>
                  </Access>
                </Contact>
                <Contact UId="201">
                  <Access UId="202" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StopBtn" /></Symbol>
                  </Access>
                </Contact>
                <SCoil UId="103">
                  <Access UId="104" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorRun" /></Symbol>
                  </Access>
                </SCoil>
                <RCoil UId="203">
                  <Access UId="204" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorRun" /></Symbol>
                  </Access>
                </RCoil>
              </Parts>
              <Wires>
                <Wire UId="401">
                  <Powerrail />
                  <NameCon UId="101" Name="in" />
                  <NameCon UId="201" Name="in" />
                </Wire>
                <Wire UId="402"><NameCon UId="101" Name="out" /><NameCon UId="103" Name="in" /></Wire>
                <Wire UId="403"><NameCon UId="103" Name="out" /></Wire>
                <Wire UId="404"><NameCon UId="201" Name="out" /><NameCon UId="203" Name="in" /></Wire>
                <Wire UId="405"><NameCon UId="203" Name="out" /></Wire>
              </Wires>
"""))))

# ── PASS 6：IdentCon 重复 UId 自动修复 ────────────────────────
# 同一 Access UId=302 被两条 Wire 引用，系统应自动修复
TEST_CASES.append(('PASS_IdentCon重复自动修复', _wrap_fc('TC_DupIdentCon', 6, _net(110, 'IdentCon重复UId自动修复', """
              <Parts>
                <Contact UId="301">
                  <Access UId="302" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StartBtn" /></Symbol>
                  </Access>
                </Contact>
                <Contact UId="303">
                </Contact>
                <Coil UId="305">
                  <Access UId="306" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorRun" /></Symbol>
                  </Access>
                </Coil>
                <Coil UId="307">
                  <Access UId="308" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorFault" /></Symbol>
                  </Access>
                </Coil>
              </Parts>
              <Wires>
                <Wire UId="401">
                  <Powerrail />
                  <NameCon UId="301" Name="in" />
                  <NameCon UId="303" Name="in" />
                </Wire>
                <Wire UId="402"><IdentCon UId="302" /><NameCon UId="301" Name="operand" /></Wire>
                <Wire UId="403"><IdentCon UId="302" /><NameCon UId="303" Name="operand" /></Wire>
                <Wire UId="404"><NameCon UId="301" Name="out" /><NameCon UId="305" Name="in" /></Wire>
                <Wire UId="405"><NameCon UId="305" Name="out" /></Wire>
                <Wire UId="406"><NameCon UId="303" Name="out" /><NameCon UId="307" Name="in" /></Wire>
                <Wire UId="407"><NameCon UId="307" Name="out" /></Wire>
              </Wires>
"""))))

# ── FAIL 1：多条 Powerrail（预期：系统抛出 ValueError 报错）─────
TEST_CASES.append(('FAIL_多Powerrail应报错', _wrap_fc('TC_MultiPowerrail', 7, _net(130, '多Powerrail错误', """
              <Parts>
                <Contact UId="21">
                  <Access UId="22" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StartBtn" /></Symbol>
                  </Access>
                </Contact>
                <SCoil UId="23">
                  <Access UId="24" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorRun" /></Symbol>
                  </Access>
                </SCoil>
                <Contact UId="31">
                  <Access UId="32" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="StopBtn" /></Symbol>
                  </Access>
                </Contact>
                <RCoil UId="33">
                  <Access UId="34" Scope="GlobalVariable">
                    <Symbol><Component Name="TestDB" /><Component Name="MotorRun" /></Symbol>
                  </Access>
                </RCoil>
              </Parts>
              <Wires>
                <Wire UId="41"><Powerrail /><NameCon UId="21" Name="in" /></Wire>
                <Wire UId="42"><NameCon UId="21" Name="out" /><NameCon UId="23" Name="in" /></Wire>
                <Wire UId="43"><NameCon UId="23" Name="out" /></Wire>
                <Wire UId="44"><Powerrail /><NameCon UId="31" Name="in" /></Wire>
                <Wire UId="45"><NameCon UId="31" Name="out" /><NameCon UId="33" Name="in" /></Wire>
                <Wire UId="46"><NameCon UId="33" Name="out" /></Wire>
              </Wires>
"""))))

# ── FAIL 2：S7-1500 专用指令（预期：系统抛出 ValueError 报错）─
TEST_CASES.append(('FAIL_S7-1500指令应报错', _wrap_fc('TC_S1500Instr', 8, _net(150, 'S7-1500专用指令', """
              <Parts>
                <Part Name="GATHER" UId="21" />
              </Parts>
              <Wires>
                <Wire UId="31"><Powerrail /><NameCon UId="21" Name="en" /></Wire>
              </Wires>
"""))))

print(f'共 {len(TEST_CASES)} 条测试用例，其中 PASS={sum(1 for n,_ in TEST_CASES if n.startswith("PASS"))}，FAIL={sum(1 for n,_ in TEST_CASES if n.startswith("FAIL"))}')

## 5. 执行测试

In [ ]:
from openness.tia_blocks import import_lad_xml_block
import traceback

results = []  # (name, expected_pass, actual_pass, msg)

for tc_name, xml in TEST_CASES:
    expected_pass = tc_name.startswith('PASS_')
    try:
        path = import_lad_xml_block(plc_sw, tc_name.replace('PASS_', '').replace('FAIL_', ''), xml)
        actual_pass = True
        msg = f'导入成功 → {path}'
    except Exception as e:
        actual_pass = False
        msg = str(e).split('\n')[0]  # 只取第一行，避免太长

    # 判断测试结果
    ok = (expected_pass == actual_pass)
    tag = '✅ OK  ' if ok else '❌ FAIL'
    exp_str = '预期通过' if expected_pass else '预期报错'
    act_str = '实际通过' if actual_pass   else '实际报错'
    print(f'{tag}  [{exp_str}/{act_str}]  {tc_name}')
    if not ok or not actual_pass:
        print(f'        {msg}')
    results.append((tc_name, expected_pass, actual_pass, ok, msg))

print()
total = len(results)
passed = sum(1 for r in results if r[3])
print(f'═══ 汇总：{passed}/{total} 通过 ═══')

## 6. 失败详情

In [ ]:
failed = [r for r in results if not r[3]]
if not failed:
    print('所有测试均通过 🎉')
else:
    print(f'以下 {len(failed)} 条测试未达预期：\n')
    for tc_name, expected_pass, actual_pass, _, msg in failed:
        exp = '通过' if expected_pass else '报错'
        act = '通过' if actual_pass   else '报错'
        print(f'  ❌  {tc_name}')
        print(f'       期望: {exp}   实际: {act}')
        print(f'       原因: {msg}')
        print()

## 7. 保存项目并关闭 TIA Portal

In [ ]:
from openness.tia_core import save_project, stop_tia_portal

save_project(project)
print('项目已保存')

# 如需保持 TIA Portal 开着以便手动检查，注释掉下一行
# stop_tia_portal(tia)
# print('TIA Portal 已关闭')